In [ ]:
# ==========================================
# STEP 1: Install dependencies
# ==========================================

!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok nest-asyncio diffusers transformers accelerate torch torchvision opencv-python

print("✅ Dependencies installed.")


# ==========================================
# STEP 2: Write app/pipeline.py
# ==========================================

import os
os.makedirs("app", exist_ok=True)

with open("app/pipeline.py", "w") as f:
    f.write(r'''
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image, ImageOps
import cv2
import numpy as np


class AEPipelineManager:
    def __init__(self):
        self.pipe = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def load_model(self):
        if self.pipe is not None:
            return

        print("Loading ControlNet Canny...")

        controlnet = ControlNetModel.from_pretrained(
            "lllyasviel/sd-controlnet-canny",
            torch_dtype=torch.float16
        )

        self.pipe = StableDiffusionControlNetPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-v1-5",
            controlnet=controlnet,
            torch_dtype=torch.float16
        )

        self.pipe.scheduler = UniPCMultistepScheduler.from_config(
            self.pipe.scheduler.config
        )

        self.pipe.to(self.device)
        print("Model loaded on GPU!" if self.device == "cuda" else "Model loaded on CPU!")

    def generate_render(self, control_image: Image.Image, prompt: str, negative_prompt: str, steps: int = 25) -> Image.Image:
        self.load_model()

        target = 512
        control_image.thumbnail((target, target), Image.Resampling.LANCZOS)

        dw = target - control_image.width
        dh = target - control_image.height
        padding = (dw // 2, dh // 2, dw - dw // 2, dh - dh // 2)
        img = ImageOps.expand(control_image, padding, fill="white")

        gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        canny_img = Image.fromarray(edges)

        result = self.pipe(
            prompt=prompt,
            image=canny_img,
            negative_prompt=negative_prompt,
            num_inference_steps=steps,
            guidance_scale=7.0,
            controlnet_conditioning_scale=1.2
        ).images[0]

        return result


ai_manager = AEPipelineManager()
''')

print("✅ pipeline.py created.")


# ==========================================
# STEP 3: Write app/main.py (2D Floor Plan Only)
# ==========================================

with open("app/main.py", "w") as f:
    f.write(r'''
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from app.pipeline import ai_manager
from PIL import Image
import io

app = FastAPI(title="BuildSure AI 2D Floor Plan Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

PROMPT = """
2D architectural floor plan, top-down orthographic view, clean CAD blueprint,
colored room zones, standard 2D furniture symbols, bed icon, sofa icon,
kitchen counter, dining table, toilet, bathtub, crisp black walls,
pure white background, flat vector style, no 3D, no perspective, no shadows.
"""

NEGATIVE = """
3D render, perspective, isometric, photorealistic, wood texture, brown floor,
tiles, exterior, sky, people, cars, shadows, depth, blurry, gray background,
muddy colors, watermark, text, logo.
"""


@app.get("/")
def home():
    return {"status": "AI 2D Floor Plan Server is running!"}


@app.post("/generate-render")
async def generate_render(
    file: UploadFile = File(...),
    prompt: str = Form(PROMPT),
    negative_prompt: str = Form(NEGATIVE),
    steps: int = Form(25)
):
    try:
        contents = await file.read()
        input_image = Image.open(io.BytesIO(contents)).convert("RGB")

        if not prompt or not prompt.strip():
            prompt = PROMPT
        if not negative_prompt or not negative_prompt.strip():
            negative_prompt = NEGATIVE

        output = ai_manager.generate_render(
            control_image=input_image,
            prompt=prompt,
            negative_prompt=negative_prompt,
            steps=steps
        )

        buf = io.BytesIO()
        output.save(buf, format="JPEG", quality=95)
        buf.seek(0)

        return StreamingResponse(buf, media_type="image/jpeg")

    except Exception as e:
        print("Generation Error:", e)
        raise HTTPException(status_code=500, detail=str(e))
''')

print("✅ main.py created.")


# ==========================================
# STEP 4: Start ngrok & FastAPI Server
# ==========================================

import nest_asyncio
import uvicorn
from pyngrok import ngrok

# REPLACE WITH YOUR NGROK AUTHTOKEN
NGROK_TOKEN = "   "

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 60)
print(f"🚀 PUBLIC BACKEND URL: {public_url}")
print("=" * 60 + "\n")

nest_asyncio.apply()

config = uvicorn.Config("app.main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()